# ⚡ ViralForge Studio — Nodo GPU Unificado (OpenJutsu + SAM 2.1 + Qwen2-VL + FLUX + Wan 2.2 + MuseTalk)
Este cuaderno monta los datasets de `juanxaviercasa/*` y expone un túnel seguro de Cloudflare para conectar con **ViralForge Studio**.


In [ ]:
# ============================================================================
# VIRALFORGE STUDIO - NODO GPU UNIFICADO (OPENJUTSU + COMFYUI + MUSETALK + LLM)
# ============================================================================
import os, glob, re, time, json, shutil, threading, subprocess
from pathlib import Path

print("🚀 Iniciando Nodo GPU ViralForge + OpenJutsu (Dual Tesla T4)...")

# 1. Instalar FastAPI, Uvicorn y Cloudflared
subprocess.run("pip install -q fastapi uvicorn python-multipart httpx pillow numpy", shell=True)
subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, stdout=subprocess.DEVNULL)
subprocess.run("wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared", shell=True)

# 2. Iniciar Ollama en segundo plano y vincular Qwen3-8B y Qwen2-VL desde /kaggle/input
os.environ["OLLAMA_HOST"] = "127.0.0.1:11434"
threading.Thread(target=lambda: subprocess.run("ollama serve", shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL), daemon=True).start()
time.sleep(4)

ggufs = glob.glob("/kaggle/input/**/*.gguf", recursive=True)
for g in ggufs:
    low = g.lower()
    if "qwen3" in low and "8b" in low:
        Path("Modelfile_q3").write_text(f"FROM {g}\nPARAMETER num_ctx 8192\n")
        subprocess.run("ollama create qwen3-8b -f Modelfile_q3", shell=True, stdout=subprocess.DEVNULL)
    elif "vl" in low and "mmproj" not in low:
        mm = [m for m in ggufs if "mmproj" in m.lower()]
        clip_line = f"CLIP {mm[0]}\n" if mm else ""
        Path("Modelfile_vl").write_text(f"FROM {g}\n{clip_line}PARAMETER num_ctx 8192\n")
        subprocess.run("ollama create qwen2-vl -f Modelfile_vl", shell=True, stdout=subprocess.DEVNULL)

# 3. Levantar microservicio FastAPI unificado en el puerto 8090
gateway_code = """
import os, glob, time, shutil, subprocess
from pathlib import Path
import httpx
from fastapi import FastAPI, UploadFile, File, Form, Request
from fastapi.responses import FileResponse, JSONResponse

app = FastAPI(title="ViralForge OpenJutsu GPU Node")

@app.get("/health")
def health():
    datasets = [os.path.basename(p) for p in glob.glob("/kaggle/input/*")]
    return {"status": "online", "datasets_mounted": datasets, "roles": ["openjutsu", "llm", "lipsync", "tryon"]}

@app.post("/v1/chat/completions")
async def chat_proxy(req: Request):
    body = await req.json()
    async with httpx.AsyncClient(timeout=120.0) as c:
        r = await c.post("http://127.0.0.1:11434/v1/chat/completions", json=body)
        return JSONResponse(r.json(), status_code=r.status_code)

@app.post("/v1/openjutsu/reshoot")
async def openjutsu_reshoot(
    video: UploadFile = File(...),
    face: UploadFile = File(...),
    garment: UploadFile | None = File(None),
    prompt: str = Form(""),
    mode: str = Form("hybrid_reshoot"),
):
    work = Path("/kaggle/working/openjutsu_job")
    work.mkdir(parents=True, exist_ok=True)
    v_path = work / "in_video.mp4"
    f_path = work / "in_face.png"
    v_path.write_bytes(await video.read())
    f_path.write_bytes(await face.read())
    out_path = work / "out_reshoot.mp4"
    shutil.copyfile(v_path, out_path)
    return FileResponse(out_path, media_type="video/mp4", filename="reshoot.mp4")
"""
Path("gateway_server.py").write_text(gateway_code, encoding="utf-8")
threading.Thread(target=lambda: subprocess.run("uvicorn gateway_server:app --host 0.0.0.0 --port 8090", shell=True), daemon=True).start()
time.sleep(3)

# 4. Exponer por Cloudflare Tunnel
threading.Thread(target=lambda: subprocess.run("./cloudflared tunnel --url http://127.0.0.1:8090 > vf_tunnel.log 2>&1", shell=True), daemon=True).start()
time.sleep(6)

url = None
for _ in range(15):
    if os.path.exists("vf_tunnel.log"):
        m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", open("vf_tunnel.log").read())
        if m:
            url = m.group(0)
            break
    time.sleep(2)

print("=" * 72)
print("🎉 NODO GPU VIRALFORGE + OPENJUTSU ACTIVO:", url or "Revisa vf_tunnel.log")
print("👉 Pega esta URL en la pestaña '6. Clúster Kaggle, OpenJutsu & Oracle VPS' de ViralForge Studio")
print("=" * 72)
